# 06 - Transformacion: Plata a Oro
## Construccion del Modelo Estrella (Star Schema)

---

### Objetivo de este notebook

Este notebook construye el **modelo dimensional** (star schema) en la capa Oro,
optimizado para consultas analiticas de alto rendimiento.

### Que es un modelo estrella?

El modelo estrella es un patron de diseno de bases de datos analiticas (OLAP)
donde una **tabla de hechos** central contiene las metricas cuantitativas y se
conecta mediante llaves foraneas a **tablas de dimensiones** que contienen los
atributos descriptivos. Esta estructura:

- Minimiza los JOINs necesarios para consultas analiticas
- Facilita la agregacion y el agrupamiento
- Es el estandar para herramientas de BI (Business Intelligence)

### Diagrama del Modelo

```
                    +---------------------+
                    |   dim_compuestos     |
                    |---------------------|
                    | PK llave_compuesto   |
                    |    id_compuesto      |
                    |    nombre_comun      |
                    |    formula_molecular |
                    |    peso_molecular    |
                    |    xlogp             |
                    |    tpsa              |
                    |    complejidad       |
                    |    clasificacion_    |
                    |    lipinski          |
                    +--------+------------+
                             |
+------------------+         |         +------------------+
|   dim_ensayos    |         |         |  dim_resultados  |
|------------------|         |         |------------------|
| PK llave_ensayo  |         |         |PK llave_resultado|
|    id_ensayo     |    +----+----+    |  resultado_      |
|    nombre_ensayo |    |  fact_  |    |  actividad       |
|    tipo_ensayo   |----+bioactiv.+----+                  |
+------------------+    +---------+    +------------------+
                        |PK llave_|
                        | bioactiv|
                        |FK llave_|
                        | compues.|
                        |FK llave_|
                        | ensayo  |
                        |FK llave_|
                        | result. |
                        | valor_  |
                        | activid.|
                        | id_gen_ |
                        | objetivo|
                        | acceso_ |
                        | objetivo|
                        |id_pubmed|
                        +---------+
```

### Tablas generadas

| Tabla | Tipo | Descripcion |
|-------|------|-------------|
| `pubchem_oro.dim_compuestos` | Dimension | Medicamentos con propiedades fisicoquimicas |
| `pubchem_oro.dim_ensayos` | Dimension | Ensayos biologicos unicos |
| `pubchem_oro.dim_resultados` | Dimension | Tipos de resultado de actividad |
| `pubchem_oro.fact_bioactividades` | Hechos | Metricas de bioactividad con llaves foraneas |

---
## 1. Importacion de Librerias y Configuracion

In [ ]:
from pyspark.sql import functions as F      # Funciones de transformacion de Spark SQL
from pyspark.sql.window import Window        # Funciones de ventana para llaves surrogadas

In [ ]:
# Configuracion de catalogo y esquemas
CATALOGO = spark.sql("SELECT current_catalog()").collect()[0][0]
ESQUEMA_PLATA = "pubchem_plata"    # Esquema de origen (datos limpios)
ESQUEMA_ORO = "pubchem_oro"        # Esquema de destino (modelo estrella)
ESQUEMA_BRONCE = "pubchem_bronce"  # Para obtener propiedades adicionales

print(f"Catalogo: {CATALOGO}")
print(f"Origen:   {ESQUEMA_PLATA}")
print(f"Destino:  {ESQUEMA_ORO}")

In [ ]:
# Cargar tabla fuente de la capa Plata (datos limpios y enriquecidos)
df_plata = spark.table(f"{ESQUEMA_PLATA}.actividades_biologicas")

# Cargar tabla de propiedades de Bronce para campos adicionales de la dimension
df_propiedades = spark.table(f"{ESQUEMA_BRONCE}.propiedades_compuestos")

# Registrar total de registros de origen
total_plata = df_plata.count()
print(f"Registros en Plata: {total_plata:,}")

---
## 2. Dimension: dim_compuestos

La dimension de compuestos contiene los **15 medicamentos** con sus propiedades
fisicoquimicas y clasificacion Lipinski. Se enriquece con campos adicionales
de la tabla de propiedades de Bronce (formula molecular, TPSA).

Se genera una **llave surrogada** (`llave_compuesto`) con `row_number()`,
que sera la llave primaria de esta dimension y la llave foranea
en la tabla de hechos.

In [ ]:
# Obtener compuestos unicos desde la capa Plata
# Se seleccionan las columnas disponibles en Plata para esta dimension
df_compuestos_plata = df_plata.select(
    "id_compuesto",            # Identificador del compuesto (CID)
    "nombre_comun",            # Nombre del medicamento en espanol
    "peso_molecular",          # Peso molecular en Daltons
    "xlogp",                   # Coeficiente de particion (lipofilicidad)
    "complejidad",             # Indice de complejidad molecular
    "clasificacion_lipinski"   # Clasificacion segun Regla de Lipinski
).distinct()  # Obtener solo registros unicos

# Preparar campos adicionales de la tabla de propiedades de Bronce
# Estos campos no se incluyeron en Plata pero son utiles para la dimension
df_props_extra = df_propiedades.select(
    F.col("CID").alias("_cid_join"),                             # Llave temporal para JOIN
    F.col("MolecularFormula").alias("formula_molecular"),         # Formula molecular
    F.col("TPSA").cast("double").alias("tpsa")                   # Area de superficie polar
)

# JOIN para combinar datos de Plata con propiedades adicionales de Bronce
df_compuestos_completo = df_compuestos_plata.join(
    df_props_extra,
    df_compuestos_plata["id_compuesto"] == df_props_extra["_cid_join"],
    "left"  # LEFT JOIN para no perder compuestos sin propiedades extra
).drop("_cid_join")  # Eliminar columna temporal

# Generar llave surrogada secuencial usando window function
# row_number() asigna un numero secuencial ordenado por id_compuesto
ventana_compuestos = Window.orderBy("id_compuesto")

dim_compuestos = df_compuestos_completo.withColumn(
    "llave_compuesto", F.row_number().over(ventana_compuestos)
).select(
    "llave_compuesto",         # PK: Llave surrogada de la dimension
    "id_compuesto",            # Identificador natural (CID de PubChem)
    "nombre_comun",            # Nombre del medicamento
    "formula_molecular",       # Formula quimica (ej: C9H8O4)
    "peso_molecular",          # Peso molecular en Daltons
    "xlogp",                   # Lipofilicidad
    "tpsa",                    # Area de superficie polar topologica
    "complejidad",             # Complejidad molecular
    "clasificacion_lipinski"   # Cumple Lipinski / No cumple Lipinski
)

# Guardar dimension como Delta Table
dim_compuestos.write.format("delta") \
    .mode("overwrite").option("overwriteSchema", "true") \
    .saveAsTable(f"{ESQUEMA_ORO}.dim_compuestos")

print(f"dim_compuestos creada: {dim_compuestos.count()} registros")
dim_compuestos.show(truncate=False)

---
## 3. Dimension: dim_ensayos

La dimension de ensayos contiene los ensayos biologicos unicos identificados
por su `id_ensayo` (AID de PubChem), con su nombre y tipo de ensayo.

Un ensayo puede tener multiples resultados (uno por cada compuesto probado),
por eso se extraen como dimension separada.

In [ ]:
# Obtener ensayos unicos desde la capa Plata
df_ensayos_unicos = df_plata.select(
    "id_ensayo",       # Identificador del ensayo (AID de PubChem)
    "nombre_ensayo",   # Nombre descriptivo del ensayo biologico
    "tipo_ensayo"      # Tipo: Screening, Confirmatory, Summary, etc.
).distinct()  # Eliminar duplicados

# Generar llave surrogada secuencial
ventana_ensayos = Window.orderBy("id_ensayo")

dim_ensayos = df_ensayos_unicos.withColumn(
    "llave_ensayo", F.row_number().over(ventana_ensayos)
).select(
    "llave_ensayo",    # PK: Llave surrogada de la dimension
    "id_ensayo",       # Identificador natural (AID de PubChem)
    "nombre_ensayo",   # Nombre del ensayo
    "tipo_ensayo"      # Tipo de ensayo
)

# Guardar dimension como Delta Table
dim_ensayos.write.format("delta") \
    .mode("overwrite").option("overwriteSchema", "true") \
    .saveAsTable(f"{ESQUEMA_ORO}.dim_ensayos")

print(f"dim_ensayos creada: {dim_ensayos.count()} registros")
dim_ensayos.show(5, truncate=True)

---
## 4. Dimension: dim_resultados

La dimension de resultados es una tabla pequena que contiene los tipos
de resultado de actividad posibles: Active, Inactive, Inconclusive, Probe, etc.

Aunque es una dimension con pocos registros, extraerla como tabla separada
facilita las consultas analiticas y permite agregar atributos descriptivos
adicionales en el futuro.

In [ ]:
# Obtener resultados de actividad unicos desde la capa Plata
df_resultados_unicos = df_plata.select("resultado_actividad").distinct()

# Generar llave surrogada secuencial
ventana_resultados = Window.orderBy("resultado_actividad")

dim_resultados = df_resultados_unicos.withColumn(
    "llave_resultado", F.row_number().over(ventana_resultados)
).select(
    "llave_resultado",        # PK: Llave surrogada de la dimension
    "resultado_actividad"     # Tipo de resultado: Active, Inactive, etc.
)

# Guardar dimension como Delta Table
dim_resultados.write.format("delta") \
    .mode("overwrite").option("overwriteSchema", "true") \
    .saveAsTable(f"{ESQUEMA_ORO}.dim_resultados")

print(f"dim_resultados creada: {dim_resultados.count()} registros")
dim_resultados.show(truncate=False)

---
## 5. Tabla de Hechos: fact_bioactividades

La tabla de hechos es la tabla central del modelo estrella. Contiene:
- **Llaves foraneas** hacia las tres dimensiones (compuestos, ensayos, resultados)
- **Metricas cuantitativas** (valor de actividad en micromolar)
- **Atributos degenerados** (id_gen_objetivo, acceso_objetivo, id_pubmed)

Los atributos degenerados son campos que no justifican una dimension propia
pero que son utiles para el analisis detallado.

Se construye mediante JOINs con las dimensiones para obtener las llaves surrogadas.

In [ ]:
# Leer dimensiones recien creadas para obtener las llaves surrogadas
dim_comp = spark.table(f"{ESQUEMA_ORO}.dim_compuestos").select(
    "llave_compuesto", "id_compuesto"
)
dim_ens = spark.table(f"{ESQUEMA_ORO}.dim_ensayos").select(
    "llave_ensayo", "id_ensayo", "nombre_ensayo", "tipo_ensayo"
)
dim_res = spark.table(f"{ESQUEMA_ORO}.dim_resultados").select(
    "llave_resultado", "resultado_actividad"
)

# Construir tabla de hechos uniendo Plata con las tres dimensiones
# INNER JOIN para garantizar integridad referencial completa
df_hechos = df_plata \
    .join(dim_comp, on="id_compuesto", how="inner") \
    .join(
        dim_ens,
        on=["id_ensayo", "nombre_ensayo", "tipo_ensayo"],
        how="inner"
    ) \
    .join(dim_res, on="resultado_actividad", how="inner")

# Generar llave surrogada para la tabla de hechos
ventana_hechos = Window.orderBy("llave_compuesto", "llave_ensayo", "llave_resultado")

fact_bioactividades = df_hechos.withColumn(
    "llave_bioactividad", F.row_number().over(ventana_hechos)
).select(
    "llave_bioactividad",   # PK: Llave surrogada de la tabla de hechos
    "llave_compuesto",      # FK: Referencia a dim_compuestos
    "llave_ensayo",         # FK: Referencia a dim_ensayos
    "llave_resultado",      # FK: Referencia a dim_resultados
    "valor_actividad_um",   # Metrica: Valor de actividad en micromolar
    "id_gen_objetivo",      # Atributo degenerado: Gen objetivo
    "acceso_objetivo",      # Atributo degenerado: Accession del objetivo
    "id_pubmed"             # Atributo degenerado: ID de publicacion
)

# Guardar tabla de hechos como Delta Table
fact_bioactividades.write.format("delta") \
    .mode("overwrite").option("overwriteSchema", "true") \
    .saveAsTable(f"{ESQUEMA_ORO}.fact_bioactividades")

# Verificar el conteo final
total_hechos = spark.table(f"{ESQUEMA_ORO}.fact_bioactividades").count()
print(f"fact_bioactividades creada: {total_hechos:,} registros")

---
## 6. Validacion de Integridad Referencial

Se verifica que **todas las llaves foraneas** en la tabla de hechos tienen
correspondencia en sus respectivas dimensiones. Esto es critico para garantizar
que las consultas analiticas con JOINs produzcan resultados correctos.

Se utiliza `LEFT_ANTI JOIN` para encontrar registros huerfanos: registros
en la tabla de hechos cuya llave foranea no existe en la dimension.

In [ ]:
# Cargar la tabla de hechos para validacion
fact = spark.table(f"{ESQUEMA_ORO}.fact_bioactividades")

print("--- Validacion de integridad referencial ---")
print()

# Validar FK hacia dim_compuestos
# LEFT_ANTI JOIN retorna registros de fact que NO tienen match en la dimension
huerfanos_comp = fact.join(
    spark.table(f"{ESQUEMA_ORO}.dim_compuestos"),
    on="llave_compuesto", how="left_anti"
).count()
print(f"  Hechos sin compuesto valido:  {huerfanos_comp}")

# Validar FK hacia dim_ensayos
huerfanos_ens = fact.join(
    spark.table(f"{ESQUEMA_ORO}.dim_ensayos"),
    on="llave_ensayo", how="left_anti"
).count()
print(f"  Hechos sin ensayo valido:     {huerfanos_ens}")

# Validar FK hacia dim_resultados
huerfanos_res = fact.join(
    spark.table(f"{ESQUEMA_ORO}.dim_resultados"),
    on="llave_resultado", how="left_anti"
).count()
print(f"  Hechos sin resultado valido:  {huerfanos_res}")

# Evaluar resultado global de la validacion
total_huerfanos = huerfanos_comp + huerfanos_ens + huerfanos_res
if total_huerfanos == 0:
    print("\nIntegridad referencial verificada correctamente.")
else:
    print(f"\nADVERTENCIA: Se detectaron {total_huerfanos} registros huerfanos.")

---
## 7. Resumen del Modelo Estrella

Se presenta un resumen consolidado de todas las tablas creadas en la capa Oro
con sus conteos de registros y columnas.

In [ ]:
# Encabezado del resumen
print("=" * 60)
print("     RESUMEN DEL MODELO ESTRELLA - CAPA ORO")
print("=" * 60)
print()

# Definicion de las tablas del modelo con su tipo
tablas_oro = [
    ("dim_compuestos", "Dimension"),
    ("dim_ensayos", "Dimension"),
    ("dim_resultados", "Dimension"),
    ("fact_bioactividades", "Hechos")
]

# Iterar sobre las tablas y mostrar metricas de cada una
for nombre_tabla, tipo in tablas_oro:
    tabla = spark.table(f"{ESQUEMA_ORO}.{nombre_tabla}")
    registros = tabla.count()         # Cantidad de registros
    columnas = len(tabla.columns)     # Cantidad de columnas
    print(f"  [{tipo:10s}] {nombre_tabla:25s} -> {registros:>10,} registros, {columnas} columnas")

# Resumen de volumetria
print()
print(f"  Origen (Plata):  {total_plata:,} registros")
print(f"  Hechos (Oro):    {total_hechos:,} registros")
print("=" * 60)